In [4]:
pip uninstall -y langchain langchain-community langchain-openai

Found existing installation: langchain 1.2.15
Uninstalling langchain-1.2.15:
  Successfully uninstalled langchain-1.2.15
Found existing installation: langchain-community 0.4.1
Uninstalling langchain-community-0.4.1:
  Successfully uninstalled langchain-community-0.4.1
Found existing installation: langchain-openai 1.1.14
Uninstalling langchain-openai-1.1.14:
  Successfully uninstalled langchain-openai-1.1.14


In [1]:
pip install langchain==0.2.16 langchain-community langchain-openai chromadb pypdf

In [ ]:
import os

# 🔐 API Key (only for testing — remove in real projects)
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"  # Set via env variable instead
# Recommended: set OPENAI_API_KEY in your terminal before running
# export OPENAI_API_KEY="your-actual-key"

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.chains.retrieval_qa.base import RetrievalQA
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# 1. Load PDF
loader = PyPDFLoader(file_path)
documents = loader.load()

# 🔍 Check API key
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("❌ Please set OPENAI_API_KEY in environment variables")

# 📄 File path
file_path =".pdf"

if not os.path.exists(file_path):
    raise FileNotFoundError(f"❌ File not found. Available files: {os.listdir()}")



# 2. Split text
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(documents)

# 3. Embeddings + DB
embeddings = OpenAIEmbeddings()
db = Chroma.from_documents(texts, embeddings, persist_directory="./chroma_db")

# 4. QA Chain
qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(model="gpt-4o-mini"),
    retriever=db.as_retriever()
)

# 5. Chat loop
print("📄 PDF Chatbot Ready! Type 'exit' to quit.\n")

while True:
    query = input("Ask: ")
    if query.lower() == "exit":
        break

    result = qa_chain.invoke({"query": query})
    print("\n🤖 Answer:", result["result"], "\n")